# AI Guardian — Violence Detection LSTM Training

**Pipeline:** YOLO-pose skeletons → Bidirectional LSTM → Binary classification (fight / non-fight)

**Dataset:** Pre-extracted skeleton .npy files from RWF-2000 (extracted via `extract_skeletons.py`)

**Output:** `violence_lstm.pt` — trained model for integration into AI Guardian

---

### How to use:
1. Run `extract_skeletons.py` locally to extract skeletons from RWF-2000
2. Upload the `skeletons/` folder as a Kaggle Dataset
3. Run this notebook with GPU accelerator enabled
4. Download `violence_lstm.pt` → put into `models/` folder of AI Guardian

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuration

In [ ]:
# ══════════════════════════════════════════════
# CHANGE THIS to your Kaggle dataset path
# ══════════════════════════════════════════════
SKELETON_DIR = "/kaggle/input/rwf2000-skeletons/skeletons"

# If running locally, uncomment:
# SKELETON_DIR = "./skeletons"

# ── Hyperparameters ──
SEQUENCE_LEN = 30       # frames per clip
INPUT_SIZE = 102        # 2 people * 17 keypoints * 3 (x,y,conf)
HIDDEN_SIZE = 128       # LSTM hidden units
NUM_LAYERS = 2          # LSTM layers
DROPOUT = 0.3
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 30
WEIGHT_DECAY = 1e-4

print(f"Skeleton dir: {SKELETON_DIR}")
print(f"Exists: {os.path.exists(SKELETON_DIR)}")

## 2. Dataset

In [ ]:
class SkeletonDataset(Dataset):
    """
    Loads pre-extracted skeleton .npy files.
    Each .npy file: shape (30, 102) = (frames, features)
    Labels: fight=1, nonfight=0
    """
    def __init__(self, root_dir: str, augment: bool = False):
        self.samples = []  # list of (path, label)
        self.augment = augment
        
        root = Path(root_dir)
        for class_dir in sorted(root.iterdir()):
            if not class_dir.is_dir():
                continue
            label = 1 if 'fight' in class_dir.name and 'non' not in class_dir.name else 0
            for npy_file in sorted(class_dir.glob('*.npy')):
                self.samples.append((str(npy_file), label))
        
        # Count
        n_fight = sum(1 for _, l in self.samples if l == 1)
        n_nonfight = sum(1 for _, l in self.samples if l == 0)
        print(f"  Loaded: {len(self.samples)} total ({n_fight} fight, {n_nonfight} nonfight)")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        seq = np.load(path).astype(np.float32)  # (30, 102)
        
        if self.augment:
            seq = self._augment(seq)
        
        return torch.FloatTensor(seq), torch.FloatTensor([label])
    
    def _augment(self, seq):
        """Data augmentation for skeleton sequences."""
        # 1. Random temporal shift (shift frames by 0-3)
        shift = np.random.randint(0, 4)
        if shift > 0:
            seq = np.roll(seq, shift, axis=0)
        
        # 2. Horizontal flip (mirror x-coordinates)
        if np.random.random() > 0.5:
            seq_reshaped = seq.reshape(SEQUENCE_LEN, 2, 17, 3)
            seq_reshaped[:, :, :, 0] = 1.0 - seq_reshaped[:, :, :, 0]  # flip x
            # Swap left/right keypoints
            swap_pairs = [(5,6), (7,8), (9,10), (11,12), (13,14), (15,16)]
            for l, r in swap_pairs:
                seq_reshaped[:, :, [l, r], :] = seq_reshaped[:, :, [r, l], :]
            # Swap person order too
            seq_reshaped[:, [0, 1], :, :] = seq_reshaped[:, [1, 0], :, :]
            seq = seq_reshaped.reshape(SEQUENCE_LEN, -1)
        
        # 3. Add small noise to coordinates
        noise = np.random.normal(0, 0.01, seq.shape).astype(np.float32)
        seq = seq + noise
        
        # 4. Random temporal speed (drop or duplicate random frames)
        if np.random.random() > 0.7:
            speed = np.random.uniform(0.8, 1.2)
            indices = np.linspace(0, SEQUENCE_LEN - 1, int(SEQUENCE_LEN * speed))
            indices = np.clip(indices, 0, SEQUENCE_LEN - 1).astype(int)
            stretched = seq[indices]
            # Resample back to SEQUENCE_LEN
            if len(stretched) != SEQUENCE_LEN:
                new_indices = np.linspace(0, len(stretched) - 1, SEQUENCE_LEN).astype(int)
                seq = stretched[new_indices]
        
        return seq

In [ ]:
# Load datasets
print("Loading training set:")
train_dir = os.path.join(SKELETON_DIR, "train")
val_dir = os.path.join(SKELETON_DIR, "val")

# If no train/val split, create one
if not os.path.exists(train_dir):
    print("No train/val split found, using root directory")
    train_dir = SKELETON_DIR
    val_dir = None

train_dataset = SkeletonDataset(train_dir, augment=True)

if val_dir and os.path.exists(val_dir):
    print("\nLoading validation set:")
    val_dataset = SkeletonDataset(val_dir, augment=False)
else:
    # Split 80/20
    print("\nSplitting 80/20 for validation...")
    total = len(train_dataset)
    val_size = int(0.2 * total)
    train_size = total - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"  Train: {train_size}, Val: {val_size}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. Model

In [ ]:
class ViolenceLSTM(nn.Module):
    """Bidirectional LSTM for skeleton-sequence violence classification."""
    
    def __init__(self, input_size=102, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout, bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    
    def forward(self, x):
        # x: (batch, seq_len=30, features=102)
        _, (h_n, _) = self.lstm(x)
        # h_n: (num_layers*2, batch, hidden) for bidirectional
        # Take last layer, both directions
        last = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (batch, hidden*2)
        return self.classifier(last).squeeze(-1)      # (batch,)


model = ViolenceLSTM(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {total_params:,} parameters ({trainable:,} trainable)")
print(model)

## 4. Training

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Training history
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
best_epoch = 0


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for sequences, labels in loader:
        sequences = sequences.to(device)    # (batch, 30, 102)
        labels = labels.squeeze().to(device) # (batch,)
        
        optimizer.zero_grad()
        logits = model(sequences)            # (batch,)
        loss = criterion(logits, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)
    
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    for sequences, labels in loader:
        sequences = sequences.to(device)
        labels = labels.squeeze().to(device)
        
        logits = model(sequences)
        loss = criterion(logits, labels)
        
        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    return total_loss / total, correct / total, all_preds, all_labels

In [ ]:
print(f"Training for {EPOCHS} epochs on {device}...")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'LR':>10}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    lr = optimizer.param_groups[0]['lr']
    marker = ''
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        marker = ' ★ BEST'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, 'best_violence_lstm.pt')
    
    print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.1%} | {val_loss:>10.4f} | {val_acc:>8.1%} | {lr:>10.6f}{marker}")

print(f"\nBest: epoch {best_epoch}, val_acc = {best_val_acc:.1%}")

## 5. Evaluation

In [ ]:
# Load best model
checkpoint = torch.load('best_violence_lstm.pt')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

# Final evaluation
val_loss, val_acc, all_preds, all_labels = eval_epoch(model, val_loader, criterion)
print(f"\nFinal Val Accuracy: {val_acc:.1%}")
print(f"Final Val Loss: {val_loss:.4f}")

# Classification report
print("\n" + classification_report(
    all_labels, all_preds,
    target_names=['Non-Fight', 'Fight'],
    digits=3
))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_title('Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[1].set_title('Accuracy', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print("Saved: training_curves.png")

## 6. Export Model

In [ ]:
# Export for AI Guardian integration
# This saves ONLY the state_dict (smaller, more portable)
export_path = "violence_lstm.pt"

torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': INPUT_SIZE,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'sequence_len': SEQUENCE_LEN,
    'val_accuracy': best_val_acc,
    'epoch': best_epoch,
}, export_path)

file_size = os.path.getsize(export_path) / 1024
print(f"Exported: {export_path} ({file_size:.0f} KB)")
print(f"Val accuracy: {best_val_acc:.1%}")
print(f"\n--- DOWNLOAD THIS FILE ---")
print(f"Put it in: AiGuardian/models/violence_lstm.pt")
print(f"The classifier.py module will auto-detect and load it.")

## 7. Quick Inference Test

In [ ]:
# Test inference on a few samples
model.eval()

print("Sample predictions:")
print(f"{'#':>3} | {'True':>6} | {'Pred':>6} | {'Score':>6} | {'Correct':>7}")
print("-" * 42)

for i in range(min(20, len(val_dataset))):
    seq, label = val_dataset[i]
    seq = seq.unsqueeze(0).to(device)
    
    with torch.no_grad():
        logit = model(seq)
        score = torch.sigmoid(logit).item()
    
    true_label = 'Fight' if label.item() > 0.5 else 'Normal'
    pred_label = 'Fight' if score > 0.5 else 'Normal'
    correct = '  OK' if true_label == pred_label else '  MISS'
    
    print(f"{i+1:>3} | {true_label:>6} | {pred_label:>6} | {score:>5.3f} | {correct}")